# 01 · Pull and clean

1. **Pull** labelled products from the ERP and taxonomy tables, and save the raw data.
2. **Clean** the descriptions: strip leading vendor codes, the regex pass (from a pattern
   file), expand abbreviations, normalize misspellings, filter to known tokens. Rows whose
   description does not survive are dropped, then marker tokens (vendor, prodcat, prodline,
   UNSPSC) are prepended.
3. **Filter** rare hierarchy paths.
4. **Save** the cleaned data and the cleaning config, datestamped. Notebooks 02-04 refer
   to this run by its stamp; the pickled cleaner travels with every trained model.
5. **Count** unique unigrams, bigrams (both orders and order-insensitive) and trigrams.

Rare-token pruning (`MIN_TOKEN_COUNT`) is not applied here: it is counted on training rows
only, inside notebooks 02-04, so held-out rows never influence which tokens survive.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))      # repo root, so `import classifier` works

DATA_DIR, CONFIG_DIR, OUTPUT_DIR = "../data", "../config", "../outputs"

import joblib
import pandas as pd

from classifier import io, cleaning

STAMP = io.stamp()
print("run stamp:", STAMP)

## Parameters

In [ ]:
# ---- source -------------------------------------------------------------------------
PULL_FROM_SQL = True               # False: reuse a raw file already in data/
RAW_FILE = None                    # e.g. "raw_data_20260930_140000.txt" when not pulling

# ---- columns --------------------------------------------------------------------------
TEXT_COL = "FullDesc"
ID_COL = "Vallen ID"
LEVEL_COLUMNS = ["Level 1 ID", "Level 2 ID", "Level 3 ID", "Level 4 ID"]
NAME_COLUMNS = ["Level 1 Name", "Level 2 Name", "Level 3 Name", "Level 4 Name"]
ORIG_COL = "FullDesc_orig"         # the untouched description is kept here

# ---- cleaning steps, in the order they run ------------------------------------------
CLEANING_FILES_DIR = "../cleaning_files"
UPPERCASE = False                  # descriptions arrive uppercase from the ERP
STRIP_LEADING_CODE = True          # drop a leading token that is not an English word
REGEX = True
PATTERN_FILE = "original.txt"      # any file in cleaning_files/patterns/
EXPAND_ABBREVS = False             # abbrevs_to_expand.csv
NORMALIZE_MISSPELLINGS = True      # to_normalize.csv
FILTER_TO_KNOWN_TOKENS = True
KNOWN_TOKEN_SOURCES = ["good_tokens", "expansions", "normalizations"]   # can add "english"
MIN_TOKEN_LENGTH = 2               # shorter tokens are dropped by the known-token filter

# ---- marker tokens, prepended after cleaning --------------------------------------
ATTACH_VENDOR = False              # VND_<Global Vendno>
ATTACH_PRODCAT = False             # CAT_<ICSP Prodcat>
ATTACH_PRODLINE = False            # PLINE_<Prodline>
ATTACH_UNSPSC = False              # UNSPSC2_46 UNSPSC4_4618 UNSPSC6_461815 UNSPSC8_46181504
UNSPSC_LEVELS = [2, 4, 6, 8]

# ---- rows -------------------------------------------------------------------------------
MIN_PATH_COUNT = 5                 # 1 = off. Stratified k-fold CV needs >= its number of splits

# ---- n-gram counts ------------------------------------------------------------------
NGRAM_MIN_OCCURRENCES = 2          # n-grams seen in fewer descriptions are not counted
SAVE_NGRAM_COUNTS = True

## 1 · Pull

In [ ]:
SQL = """
with
    base as (
        select s.id as `ID`,
               s.erp_cono as `Cono`,
               s.erp_product_number as `Vallen ID`,
               replace(i.descrip, ';', ' ') as `FullDesc`,
               s.category_l4_id,
               s.global_vendor_number as `Global Vendno`,
               i.prodcat as `ICSP Prodcat`,
               i.zzunspsc as `UNSPSC`,
               concat(s.erp_cono, w.prodline) as `Prodline`
        from prd_bronze.stibo_ftp.sku s
        inner join prd_bronze.openedge.icsp i
            on s.erp_product_number = i.prod
        left join prd_bronze.openedge.icsw w
            on s.erp_product_number = w.prod
            and w.st_row_current = 1
            and w.cono = 1
            and w.whse = 'zdat'
        where s.st_row_current = 1
            and s.category_l4_id is not null
            and i.st_row_current = 1
            and i.cono in (1, 55)
            and i.descrip is not null
    ),

    hierarchy as (
        select
            l1.id   as `Level 1 ID`,
            l1.name as `Level 1 Name`,
            l2.id   as `Level 2 ID`,
            l2.name as `Level 2 Name`,
            l3.id   as `Level 3 ID`,
            l3.name as `Level 3 Name`,
            l4.id   as `Level 4 ID`,
            l4.name as `Level 4 Name`
        from prd_bronze.stibo_ftp.hierarchy l1
        left join prd_bronze.stibo_ftp.hierarchy l2
            on l2.parent_id = l1.id and l2.level = 2 and l2.st_row_current = 1
        left join prd_bronze.stibo_ftp.hierarchy l3
            on l3.parent_id = l2.id and l3.level = 3 and l3.st_row_current = 1
        left join prd_bronze.stibo_ftp.hierarchy l4
            on l4.parent_id = l3.id and l4.level = 4 and l4.st_row_current = 1
        where l1.level = 1
            and l1.st_row_current = 1
    )

select *
from base b
inner join hierarchy h
    on b.category_l4_id = h.`Level 4 ID`
where h.`Level 4 ID` is not null
"""

In [ ]:
if PULL_FROM_SQL:
    raw = io.spark_to_pandas(spark.sql(SQL))          # noqa: F821 (spark is provided by Databricks)
    RAW_FILE = f"raw_data_{STAMP}.txt"
    io.write_table(raw, f"{DATA_DIR}/{RAW_FILE}")
else:
    raw = io.read_table(f"{DATA_DIR}/{RAW_FILE}")

print(f"{len(raw):,} rows, {raw.shape[1]} columns from {RAW_FILE}")
raw.head()

## 2 · The cleaner

In [ ]:
markers = cleaning.build_markers(vendor=ATTACH_VENDOR, prodcat=ATTACH_PRODCAT,
                                 prodline=ATTACH_PRODLINE, unspsc=ATTACH_UNSPSC,
                                 unspsc_levels=UNSPSC_LEVELS)
cleaner = cleaning.build_cleaner(
    cleaning_files_dir=CLEANING_FILES_DIR, pattern_file=PATTERN_FILE, uppercase=UPPERCASE,
    strip_leading_code=STRIP_LEADING_CODE, regex=REGEX, expand_abbrevs=EXPAND_ABBREVS,
    normalize_misspellings=NORMALIZE_MISSPELLINGS, filter_to_known_tokens=FILTER_TO_KNOWN_TOKENS,
    known_token_sources=KNOWN_TOKEN_SOURCES, min_token_length=MIN_TOKEN_LENGTH, markers=markers,
)
print(cleaner.describe())

In [ ]:
# The resolved regex patterns, in the order they run
pd.set_option("display.max_colwidth", 200)
pd.DataFrame(cleaner.pattern_steps, columns=["kind", "pattern", "replacement"])

In [ ]:
# A sample through each step, before running on everything
cleaner.trace(raw[TEXT_COL].sample(15, random_state=0))

## 3 · Clean and filter

In [ ]:
steps = {"raw rows": len(raw)}
work = raw[raw[TEXT_COL].str.strip() != ""]
steps["with a description"] = len(work)
work = work[(work[LEVEL_COLUMNS] != "").all(axis=1)]
steps["with every level labelled"] = len(work)

cleaned = cleaner.clean_frame(work, TEXT_COL, orig_col=ORIG_COL, drop_empty=True)
steps["description survives cleaning"] = len(cleaned)

cleaned = cleaning.drop_rare_paths(cleaned, LEVEL_COLUMNS, MIN_PATH_COUNT)
steps[f"path seen >= {MIN_PATH_COUNT} times"] = len(cleaned)

row_counts = pd.Series(steps, name="rows").to_frame()
row_counts["dropped"] = -row_counts["rows"].diff().fillna(0).astype(int)
display(row_counts)

n_dup = int(cleaned[ID_COL].duplicated().sum())
if n_dup:
    print(f"note: {n_dup:,} rows share an {ID_COL} with another row; test sets and "
          "blacklists act on every row with a listed ID")

In [ ]:
# Before and after
cleaned[[ORIG_COL, TEXT_COL]].sample(min(25, len(cleaned)), random_state=0)

## 4 · Save

In [ ]:
CLEANED_FILE = f"cleaned_data_{STAMP}.txt"
io.write_table(cleaned, f"{DATA_DIR}/{CLEANED_FILE}")
joblib.dump(cleaner, f"{CONFIG_DIR}/cleaner_{STAMP}.joblib", compress=3)

cleaning_config = {
    "stamp": STAMP,
    "raw_file": RAW_FILE,
    "cleaned_file": CLEANED_FILE,
    "cleaner_file": f"cleaner_{STAMP}.joblib",
    "text_col": TEXT_COL, "orig_col": ORIG_COL, "id_col": ID_COL,
    "level_columns": LEVEL_COLUMNS, "name_columns": NAME_COLUMNS,
    "settings": cleaner.settings,
    "markers": cleaner.markers,
    "pattern_steps": cleaner.pattern_steps,     # resolved, so later file edits cannot change this run
    "min_path_count": MIN_PATH_COUNT,
    "row_counts": steps,
}
io.save_json(cleaning_config, f"{CONFIG_DIR}/cleaning_{STAMP}.json")
print(f"\nCLEANING_STAMP = \"{STAMP}\"   <- use this in notebooks 02-04")

## 5 · N-gram counts

Document counts, tokenized the way the vectorizer does (lowercase, two or more
characters). Bigrams are counted both as written and order-insensitive — the second is
what the model sees with `unordered_bigrams=True`.

In [ ]:
ngram_table, ngram_counts = cleaning.ngram_summary(cleaned[TEXT_COL], NGRAM_MIN_OCCURRENCES)
ngram_table

In [ ]:
# Bigrams written both ways round: each pair with the count in each order
ordered = cleaning.count_ngrams(cleaned[TEXT_COL], 2)          # every bigram, unfiltered
pairs = pd.DataFrame({"bigram": ordered.index, "count": ordered.to_numpy()})
pairs["reversed"] = pairs["bigram"].map(lambda b: " ".join(b.split()[::-1]))
pairs["reversed_count"] = pairs["reversed"].map(ordered).fillna(0).astype(int)
both = pairs[(pairs["reversed_count"] > 0) & (pairs["bigram"] < pairs["reversed"])].copy()
both["combined"] = both["count"] + both["reversed_count"]
both = both.sort_values("combined", ascending=False)
both = both[both["combined"] >= NGRAM_MIN_OCCURRENCES]
print(f"{len(both):,} bigrams appear in both orders")
both.head(30)

In [ ]:
for name, counts in ngram_counts.items():
    print(f"\n{name}: top 15")
    print(counts.head(15).to_string())

if SAVE_NGRAM_COUNTS:
    out_dir = f"{OUTPUT_DIR}/ngrams_{STAMP}"
    os.makedirs(out_dir, exist_ok=True)
    ngram_table.to_csv(f"{out_dir}/summary.csv", index=False)
    for name, counts in ngram_counts.items():
        fname = name.replace(" ", "_").replace("(", "").replace(")", "")
        counts.rename("doc_count").rename_axis("ngram").to_csv(f"{out_dir}/{fname}.csv")
    both.to_csv(f"{out_dir}/bigrams_both_orders.csv", index=False)
    print(f"\nwrote n-gram counts to {out_dir}")